# Topic: File Pointers - Cursor navigation, seek() and tell() parameters, and text mode constraints
 
## 1. THE FILE POINTER (CURSOR)
- **File Pointer**: An integer offset maintained by the OS representing the current byte position in the file where the next read or write operation will begin.
- **`f.tell()`**: Returns the current position of the file pointer as an integer representing the offset bytes from the start of the file.
 
## 2. NAVIGATION: seek()
- **`f.seek(offset, whence=0)`**:
  - Shifts the file pointer by `offset` bytes.
  - **`whence` Options** (from standard `os` module):
    - **`0` (`os.SEEK_SET`)**: Offset is relative to the **beginning** of the file (Default).
    - **`1` (`os.SEEK_CUR`)**: Offset is relative to the **current** cursor position.
    - **`2` (`os.SEEK_END`)**: Offset is relative to the **end** of the file (usually negative values to move backwards).
 
## 3. TEXT MODE CONSTRAINTS
- **Rule**: In text mode (`t`), seeking relative to current (`whence=1`) or end (`whence=2`) is **restricted to an offset of 0**.
- **Reason**: Multi-byte character encodings (like UTF-8, where characters can take 1 to 4 bytes) make matching byte offsets to character counts inconsistent. Seeking non-zero offsets from current/end in text mode raises an error or leads to invalid character boundaries.
- To seek arbitrary offsets in text mode, you must use `whence=0` with values returned directly by `tell()`. For arbitrary byte seeks, open the file in **binary mode** (`b`).
 
---


In [ ]:
import os

pointer_file = "pointer_demo.txt"

# Create a sample file
with open(pointer_file, "wb") as f:
    f.write(b"abcdefghijklmnopqrstuvwxyz")  # 26 bytes

# 1. Using tell() and seek() in Binary Mode (no constraints)
print("--- Binary Seek/Tell Operations ---")
with open(pointer_file, "rb") as f:
    print(f"Start cursor: {f.tell()}")  # Expected: 0
    
    # Read 5 bytes
    print(f"Read 5 bytes: {f.read(5)}")  # b'abcde'
    print(f"Cursor after read: {f.tell()}")  # Expected: 5
    
    # Seek from start (whence=0)
    f.seek(10, os.SEEK_SET)
    print(f"Seek(10, SET) cursor: {f.tell()}")  # Expected: 10
    print(f"Read 1: {f.read(1)}")  # b'k'
    
    # Seek from current (whence=1) - move forward by 2 bytes
    f.seek(2, os.SEEK_CUR)
    print(f"Seek(2, CUR) cursor: {f.tell()}")  # Expected: 13 (10 + 1 + 2)
    print(f"Read 1: {f.read(1)}")  # b'n'
    
    # Seek from end (whence=2) - move backward 5 bytes
    f.seek(-5, os.SEEK_END)
    print(f"Seek(-5, END) cursor: {f.tell()}")  # Expected: 21 (26 - 5)
    print(f"Read remaining: {f.read()}")  # b'vwxyz'



In [ ]:
# 2. Text Mode Constraints Verification
print("\n--- Text Mode Seek Constraints ---")
with open(pointer_file, "r", encoding="utf-8") as f:
    # Seek 0 from current/end is allowed in text mode
    f.seek(0, os.SEEK_END)
    print(f"Seek 0 from END success, cursor: {f.tell()}")
    
    try:
        # Seek non-zero from end is a violation in text mode!
        f.seek(-5, os.SEEK_END)
    except io.UnsupportedOperation as e:
        # Note: on some platforms this might raise OSError or UnsupportedOperation
        print(f"Caught expected Exception: {e}")



In [ ]:
# 3. Clean up
if os.path.exists(pointer_file):
    os.remove(pointer_file)
